# 真实数据案例：研究问题、可复现分析与敏感性分析

来自《生物医学网络科学》（Biomedical Network Science）。数据文件放在与本笔记本同级的 `data/` 文件夹中。

# 真实数据案例：给“疾病基因在相互作用组中聚集”做一次多元宇宙分析

**问题**：在 [疾病模块](network-medicine.disease-modules.html) 分析里，“某疾病的基因形成显著模块”这个结论依赖一串分析选择：相互作用网络取多高的置信度阈值、疾病基因取多高的证据阈值、零模型是否保持度、随机化做多少次、用哪个随机种子。本案例把这些选择全部列出来，在所有组合（“宇宙”）下重算一遍，看哪些疾病的结论稳健、哪些只在特定选择下成立。

**数据来源**（与 [网络医学的真实数据案例](network-medicine.html) 相同）

- STRING v12.0 人类物理相互作用子网络，综合分数 ≥ 700（CC BY 4.0；Szklarczyk D, et al. *Nucleic Acids Res* 2023;51:D638–D646）。
- Open Targets Platform（数据版本 26.09）11 种疾病的遗传关联分数 ≥ 0.5 的基因（CC0 1.0；Buniello A, et al. *Nucleic Acids Res* 2025;53:D1467–D1475）。

对应的知识页：[可复算的分析](reproducible-research.reproducible-analysis.html)、[计算环境](reproducible-research.computational-environment.html)、[敏感性分析](reproducible-research.sensitivity-analysis.html)、[多元宇宙分析](reproducible-research.multiverse-analysis.html)、[随机性与稳健性](reproducible-research.randomness-robustness.html)。

## 1 数据来源与计算环境：先记录，再分析

可复算的第一步是把“用了什么”写下来：数据文件的校验和（文件内容变了，校验和就会变）、软件版本、全部参数和随机种子集中放在一个配置里。

In [ ]:
import hashlib
import sys
import platform
import numpy as np
import pandas as pd
import scipy
import matplotlib
import matplotlib.pyplot as plt
from itertools import product
from scipy import sparse
from scipy.sparse.csgraph import connected_components

plt.rcParams["font.sans-serif"] = ["Noto Sans CJK SC", "PingFang SC", "Heiti SC", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

FILES = ["data/string_human_physical_v12_s700.csv", "data/opentargets_disease_genes_ga.csv"]
for f in FILES:
    with open(f, "rb") as fh:
        print("%-45s sha256 = %s…" % (f, hashlib.sha256(fh.read()).hexdigest()[:16]))
print("Python %s | numpy %s | scipy %s | pandas %s | matplotlib %s | %s"
      % (sys.version.split()[0], np.__version__, scipy.__version__, pd.__version__, matplotlib.__version__, platform.machine()))

CONFIG = {
    "string_threshold": [700, 800, 900],      # 相互作用组：STRING 综合分数阈值
    "ga_threshold": [0.5, 0.6, 0.7, 0.8],     # 疾病基因：遗传关联分数阈值
    "null_model": ["度匹配", "均匀"],         # 零模型
    "n_random": 200,                          # 每个宇宙的随机化次数
    "seed": 15,                               # 主分析的随机种子
    "min_genes": 10,                          # 疾病基因少于此数的宇宙不计算
    "alpha": 0.05,
}
print(CONFIG)

## 2 读入与清洗

每个 STRING 阈值都要重新取最大连通分支（LCC），因为阈值越高，网络越稀疏，落在 LCC 外的蛋白越多。疾病基因只保留在当前 LCC 中的部分。

In [ ]:
ppi = pd.read_csv(FILES[0])
dg = pd.read_csv(FILES[1])
diseases = list(dg.drop_duplicates("disease_id").disease)

def build_network(th):
    e = ppi[ppi.score >= th]
    nodes = pd.Index(sorted(set(e.protein1) | set(e.protein2)))
    ix = pd.Series(np.arange(len(nodes)), index=nodes)
    i, j = ix[e.protein1].values, ix[e.protein2].values
    A = sparse.coo_matrix((np.ones(len(i)), (i, j)), shape=(len(nodes),) * 2)
    A = ((A + A.T) > 0).astype(np.int8).tocsr()
    _, comp = connected_components(A, directed=False)
    keep = np.where(comp == np.bincount(comp).argmax())[0]
    G = A[keep][:, keep].tocsr()
    return G, pd.Series(np.arange(len(keep)), index=nodes[keep])

def degree_bins(deg, min_size=100):
    order = np.argsort(deg, kind="stable")
    bins, cur = [], []
    for pos, v in enumerate(order):
        cur.append(v)
        if len(cur) >= min_size and not (pos + 1 < len(order) and deg[order[pos + 1]] == deg[v]):
            bins.append(np.array(cur)); cur = []
    if cur:
        bins[-1] = np.concatenate([bins[-1], cur])
    bin_of = np.empty(len(deg), dtype=int)
    for k, arr in enumerate(bins):
        bin_of[arr] = k
    return bins, bin_of

NET = {}
for th in CONFIG["string_threshold"]:
    G, gid = build_network(th)
    deg = np.asarray(G.sum(1)).ravel()
    NET[th] = (G, gid, deg) + degree_bins(deg)
    print("STRING ≥ %d：LCC %5d 个节点、%6d 条边，平均度 %.1f" % (th, G.shape[0], G.nnz // 2, deg.mean()))

## 3 构网与核心指标：每个宇宙中的模块大小与 $z$ 分数

指标与网络医学案例相同：疾病基因诱导子图的最大连通分支大小 $S_\mathrm{LCC}$，与随机基因集比较得到 $z$ 分数和经验 $p$ 值 $p=(1+\#\{S^\mathrm{rand}\ge S\})/(B+1)$。

In [ ]:
def lcc_size(G, S):
    if len(S) == 0:
        return 0
    _, c = connected_components(G[S][:, S], directed=False)
    return np.bincount(c).max()

def random_set(S, null, N, bins, bin_of, rng):
    if null == "均匀":
        return rng.choice(N, len(S), replace=False)
    out, used = [], set()
    for s in S:
        pool = bins[bin_of[s]]
        c = rng.choice(pool)
        while c in used:
            c = rng.choice(pool)
        used.add(c); out.append(c)
    return np.array(out)

def module_test(th, ga, disease, null, B, seed):
    G, gid, deg, bins, bin_of = NET[th]
    g = dg[(dg.disease == disease) & (dg.ga_score >= ga)].gene
    S = gid[[x for x in g if x in gid.index]].values
    if len(S) < CONFIG["min_genes"]:
        return len(S), np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    obs = lcc_size(G, S)
    null_d = np.array([lcc_size(G, random_set(S, null, G.shape[0], bins, bin_of, rng)) for _ in range(B)])
    sd = null_d.std()
    z = (obs - null_d.mean()) / sd if sd > 0 else np.nan
    return len(S), obs, z, (1 + (null_d >= obs).sum()) / (B + 1)

rows = []
for th, ga, null in product(CONFIG["string_threshold"], CONFIG["ga_threshold"], CONFIG["null_model"]):
    for d in diseases:
        n, obs, z, pv = module_test(th, ga, d, null, CONFIG["n_random"], CONFIG["seed"])
        rows.append({"STRING": th, "GA": ga, "零模型": null, "疾病": d, "基因数": n, "LCC": obs, "z": z, "p": pv})
uni = pd.DataFrame(rows)
print("宇宙数（不含疾病维度）：%d；疾病 × 宇宙 共 %d 个结果，其中 %d 个因基因少于 %d 而未计算"
      % (len(uni) // len(diseases), len(uni), uni.z.isna().sum(), CONFIG["min_genes"]))

## 4 与零模型比较：每种疾病在多少个宇宙里“显著”

对每种疾病，统计在全部宇宙中 $p<0.05$ 的比例，并分零模型列出 $z$ 的中位数。比例接近 1 的结论稳健；比例在中间的结论取决于分析选择，应当如实报告这种依赖。

In [ ]:
uni["显著"] = uni.p < CONFIG["alpha"]
summ = uni.groupby("疾病").agg(宇宙数=("z", "count"), 显著比例=("显著", "mean"),
                               z_最小=("z", "min"), z_中位数=("z", "median"), z_最大=("z", "max"))
by_null = uni.pivot_table(index="疾病", columns="零模型", values="z", aggfunc="median")
summ = summ.join(by_null.add_prefix("z 中位数·")).sort_values("显著比例")
pd.set_option("display.width", 200)
print(summ.round(2).to_string())

**分岔花园**：如果每种疾病都只报告“最好看”的那个宇宙，会得到什么印象？把每种疾病的最大 $z$ 与预先指定的主分析（STRING ≥ 700、GA ≥ 0.7、度匹配）对比。

In [ ]:
main = uni[(uni.STRING == 700) & (uni.GA == 0.7) & (uni["零模型"] == "度匹配")].set_index("疾病")
best = uni.loc[uni.groupby("疾病").z.idxmax()].set_index("疾病")
cmp_ = pd.DataFrame({"主分析 z": main.z, "主分析 p": main.p, "最好看的 z": best.z,
                     "对应选择": best.STRING.astype(str) + " / GA " + best.GA.astype(str) + " / " + best["零模型"]})
print(cmp_.round(3).to_string())
print("\n主分析中显著的疾病：%d / %d；“各取最好”时显著的疾病：%d / %d"
      % ((main.p < 0.05).sum(), len(main), (best.p < 0.05).sum(), len(best)))

## 5 随机性：同一个宇宙，换随机种子会怎样？

取一个处在边界上的疾病（主分析中 $p$ 最接近 0.05 的那个），在主分析宇宙中用 20 个不同种子，分别做 $B=100$ 和 $B=1000$ 次随机化，比较 $z$ 与 $p$ 的波动。$B=1000$ 的 20 次重复在浏览器里需要十几秒。

In [ ]:
edge_d = (main.p - 0.05).abs().idxmin()
print("边界疾病：", edge_d, "（主分析 p = %.3f）" % main.loc[edge_d, "p"])
seed_rows = []
for B in (100, 1000):
    for s in range(20):
        _, _, z, pv = module_test(700, 0.7, edge_d, "度匹配", B, 1000 + s)
        seed_rows.append({"B": B, "seed": s, "z": z, "p": pv})
seeds = pd.DataFrame(seed_rows)
print(seeds.groupby("B").agg(z_均值=("z", "mean"), z_标准差=("z", "std"), p_最小=("p", "min"), p_最大=("p", "max"),
                             p小于005的比例=("p", lambda v: (v < 0.05).mean())).round(3).to_string())

## 6 作图：规格曲线（specification curve）

上：所有“疾病 × 宇宙”结果的 $z$ 从小到大排列，实心点为 $p<0.05$；下：每个结果对应的分析选择。这种图把“结论依赖哪些选择”一次性摊开，比只报告一个数字诚实得多。

In [ ]:
sc = uni.dropna(subset=["z"]).sort_values("z").reset_index(drop=True)
fig, ax = plt.subplots(2, 1, figsize=(12, 6.5), sharex=True, gridspec_kw={"height_ratios": [2.2, 1.6]})
x = np.arange(len(sc))
ax[0].scatter(x[sc["显著"]], sc.z[sc["显著"]], s=10, color="#3b6ea8", label="p < 0.05")
ax[0].scatter(x[~sc["显著"]], sc.z[~sc["显著"]], s=10, facecolors="none", edgecolors="#c0392b", label="p ≥ 0.05")
ax[0].axhline(1.96, color="grey", ls="--", lw=1)
ax[0].set_ylabel("模块 LCC 的 z 分数")
ax[0].set_yscale("symlog", linthresh=2)
ax[0].legend(fontsize=8, loc="upper left")
ax[0].set_title("同一问题的 %d 个分析结果：多数疾病在各种选择下都显著，少数疾病的结论取决于选择" % len(sc))

labels, ypos = [], 0
for col, vals in [("零模型", CONFIG["null_model"]), ("GA", CONFIG["ga_threshold"]), ("STRING", CONFIG["string_threshold"])]:
    for v in vals:
        m = (sc[col] == v).values
        ax[1].scatter(x[m], np.full(m.sum(), ypos), s=3, marker="|", color="#333333")
        labels.append("%s = %s" % (col, v)); ypos += 1
ax[1].set_yticks(range(len(labels))); ax[1].set_yticklabels(labels, fontsize=8)
ax[1].set_xlabel("按 z 排序的结果")
plt.tight_layout()

## 7 解读与局限

**结果说明了什么**

- 阿尔茨海默病、克罗恩病和类风湿关节炎在全部 24 个宇宙中都显著，银屑病和溃疡性结肠炎在 23 个中显著：“这些疾病的遗传关联基因在相互作用组中聚集”是稳健的结论，可以报告，同时附上规格曲线。
- 双相障碍在任何宇宙中都不显著，精神分裂症只在约四分之一的宇宙中显著，2 型糖尿病和系统性红斑狼疮在一半左右的宇宙中显著。对这类疾病，合适的表述是“证据取决于基因集定义与零模型”，而不是挑一个显著的宇宙报告。2 型糖尿病尤其依赖零模型：度匹配基线下 $z$ 的中位数约 4.9，均匀基线下只有约 0.8。
- 主分析与“各取最好”的对比显示了分岔花园的代价：显著的疾病数只从 9 种变成 10 种，但效应量被大幅抬高，例如 2 型糖尿病的 $z$ 从约 3.2 变成约 21，银屑病从约 6.9 变成约 25。防止这一点的办法是预先指定主分析（[预注册](reproducible-research.preregistration.html)），把其余宇宙作为敏感性分析完整报告。
- 随机种子：在边界疾病上，$B=100$ 时不同种子给出的 $z$ 标准差约 0.7，$B=1000$ 时约 0.15。经验 $p$ 值的最小可能值是 $1/(B+1)$，所以 $B=200$ 时最显著的结果都显示为 $p\approx0.005$。随机化次数要和需要的精度匹配，并报告种子。

**局限**

- 这里的“宇宙”只覆盖了三个维度。真实分析还有更多选择：STRING 用物理子网络还是全部关联、度分箱的粒度、是否去掉 HLA 区域等高度相关的基因簇、用 LCC 还是平均最短距离作指标。
- 各宇宙共享同一份数据，结果高度相关；“显著比例”不是独立重复的成功率，不能当作概率解读。
- 稳健不等于正确：所有宇宙都依赖同一个不完整、偏向研究充分蛋白的相互作用组，系统性偏倚不会因为换阈值而消失。
- 本案例的计算环境记录在第 1 步；在浏览器（Pyodide）与本机运行时，库版本不同，随机数生成器相同（numpy 的 PCG64），但个别数值仍可能有细微差别，这正是要记录版本的原因。